In [118]:
import json
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt

import pandas as pd
from pandas.tseries.offsets import MonthEnd
import numpy as np

# Payment offsets are days before the target month end.
# The duplicate installation -49 day milestone is intentionally kept as two 25% payments.


In [119]:

df_path = "../data/client_provided_data/ck_dashboard_demo.xlsx"
df=pd.read_excel(df_path, sheet_name='Sheet1')

PermissionError: [Errno 13] Permission denied: '../data/client_provided_data/ck_dashboard_demo.xlsx'

In [ ]:


with open('../data/assumptions.json') as _f:
    ASSUMPTIONS = json.load(_f)

PAYMENT_SCHEDULE = pd.DataFrame(ASSUMPTIONS["payment_schedule"])


def collapse_target_sockets_to_latest_month(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a new table where, for each row, all target_sockets_X values are summed
    into the latest target_sockets_X column that has a non-zero / non-null value.
    
    Example
    -------
    target_sockets_1 = 10, target_sockets_2 = 10, target_sockets_3 = 10
    -->
    target_sockets_1 = 0, target_sockets_2 = 0, target_sockets_3 = 30
    """
    df_new = df.copy()

    target_cols = [col for col in df_new.columns if re.fullmatch(r"target_sockets_\d+", col)]
    if not target_cols:
        raise ValueError("No target socket columns found. Expected columns like 'target_sockets_1'.")

    target_cols = sorted(target_cols, key=lambda col: int(col.rsplit("_", 1)[1]))

    # Ensure numeric
    df_new[target_cols] = df_new[target_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    arr = df_new[target_cols].to_numpy(dtype=float)

    # Treat only non-zero values as "active"
    active_mask = arr != 0

    # Row total across all target_sockets_X
    row_sums = arr.sum(axis=1)

    # Identify rows with at least one active month
    has_active = active_mask.any(axis=1)

    # Find last active month position per row
    # Reverse columns, find first True from right, then convert back to original index
    last_active_pos = np.where(
        has_active,
        arr.shape[1] - 1 - active_mask[:, ::-1].argmax(axis=1),
        -1
    )

    # Build collapsed array
    collapsed = np.zeros_like(arr, dtype=float)
    valid_rows = np.where(has_active)[0]
    collapsed[valid_rows, last_active_pos[valid_rows]] = row_sums[valid_rows]

    # Write back to new dataframe
    df_new[target_cols] = collapsed

    # Optional: keep target_sockets consistent with monthly totals
    if "target_sockets" in df_new.columns:
        df_new["target_sockets"] = row_sums

    return df_new


def calculate_incurred_capex_by_month(
    df: pd.DataFrame,
    target_month_1: str | pd.Timestamp,
    group_cols=("region_name", "contract_name", "work_package_name"),
):
    """Calculate incurred CAPEX by month and cost type, after collapsing each row's
    target_sockets_X values into the latest active month.

    Parameters
    ----------
    df:
        Work-package DataFrame with capex cost-per-socket columns and target_sockets_N columns.
    target_month_1:
        Calendar month represented by target_sockets_1, for example "2026-01-01".
        target_sockets_2 is the following month, target_sockets_3 the month after, etc.
    group_cols:
        Dimensions to keep in detailed and grouped outputs.

    Returns
    -------
    df_collapsed:
        New table after collapsing target_sockets_X into the latest active month.
    detail:
        One row per work package, target month, cost type, and scheduled payment.
    monthly_by_type:
        Monthly incurred cost by selected group columns and cost type.
    monthly_pivot:
        Monthly incurred cost with one column per cost type plus total_capex.
    """
    # Step 1: create the new table
    df_collapsed = collapse_target_sockets_to_latest_month(df)

    group_cols = list(group_cols)

    target_cols = [col for col in df_collapsed.columns if re.fullmatch(r"target_sockets_\d+", col)]
    if not target_cols:
        raise ValueError("No target socket columns found. Expected columns like 'target_sockets_1'.")

    target_cols = sorted(target_cols, key=lambda col: int(col.rsplit("_", 1)[1]))
    cost_cols = sorted(PAYMENT_SCHEDULE["cost_column"].unique())

    required_cols = set(group_cols) | set(target_cols) | set(cost_cols)
    missing_cols = sorted(required_cols - set(df_collapsed.columns))
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    start_month = pd.Timestamp(target_month_1).to_period("M").to_timestamp()

    socket_long = df_collapsed.melt(
        id_vars=group_cols + cost_cols,
        value_vars=target_cols,
        var_name="target_socket_column",
        value_name="_target_sockets_value",
    )

    socket_long["target_sockets"] = socket_long["_target_sockets_value"].fillna(0)
    socket_long = socket_long.drop(columns="_target_sockets_value")
    socket_long = socket_long[socket_long["target_sockets"] != 0].copy()

    socket_long["target_month_number"] = (
        socket_long["target_socket_column"].str.extract(r"(\d+)$").astype(int)
    )

    socket_long["target_month_end"] = socket_long["target_month_number"].apply(
        lambda month_no: start_month + pd.DateOffset(months=month_no - 1) + MonthEnd(0)
    )

    detail = socket_long.merge(PAYMENT_SCHEDULE, how="cross")

    detail["cost_per_socket"] = 0.0
    for cost_column in cost_cols:
        mask = detail["cost_column"].eq(cost_column)
        detail.loc[mask, "cost_per_socket"] = detail.loc[mask, cost_column]

    detail["incurred_date"] = detail["target_month_end"] + pd.to_timedelta(
        detail["offset_days"],
        unit="D",
    )

    detail["incurred_month"] = detail["incurred_date"].dt.to_period("M").dt.to_timestamp("M")

    detail["incurred_cost"] = (
        detail["target_sockets"] * detail["cost_per_socket"] * detail["payment_pct"]
    )

    detail_cols = group_cols + [
    "target_socket_column",
    "target_month_number",
    "target_month_end",
    "cost_type",
    "payment_installment",
    "offset_days",
    "payment_pct",
    "target_sockets",
    "cost_per_socket",
    "incurred_date",
    "incurred_month",
    "incurred_cost",
    ]

    detail = detail[detail_cols].sort_values(
        group_cols + ["incurred_month", "cost_type", "offset_days"]
    )

    monthly_by_type = (
        detail.groupby(group_cols + ["incurred_month", "cost_type"], as_index=False)["incurred_cost"]
        .sum()
        .sort_values(group_cols + ["incurred_month", "cost_type"])
    )

    monthly_pivot = (
        monthly_by_type.pivot_table(
            index=group_cols + ["incurred_month"],
            columns="cost_type",
            values="incurred_cost",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
        .rename_axis(columns=None)
    )

    for col in ["bom", "connection", "installation"]:
        if col not in monthly_pivot.columns:
            monthly_pivot[col] = 0.0

    monthly_pivot["total_capex"] = monthly_pivot[["bom", "connection", "installation"]].sum(axis=1)

    monthly_pivot = monthly_pivot[
        group_cols + ["incurred_month", "bom", "connection", "installation", "total_capex"]
    ]

    return df_collapsed, detail, monthly_by_type, monthly_pivot

In [ ]:
df_collapsed, detail, monthly_by_type, monthly_pivot = calculate_incurred_capex_by_month(
    df=df,
    target_month_1="2026-01-01"
)

In [ ]:
# df_collapsed.to_excel("../data/client_provided_data/df_collapsed.xlsx", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_incurred_cost_by_month_and_installment(
    detail: pd.DataFrame,
    cost_type: str,
    end_month: str = "2026-12-31"
):
    """
    Plot incurred cost by month, stacked by payment installment.

    Changes from previous version:
    1. Uses payment_installment instead of offset_days, so installments remain separate
       even if they share the same offset_days.
    2. Legend shows 'Instalment X'.
    3. X-axis tick labels use month-year format, e.g. Jan-2026.
    """

    plot_df = detail.copy()
    plot_df["incurred_month"] = pd.to_datetime(plot_df["incurred_month"])

    end_month = pd.Timestamp(end_month).to_period("M").to_timestamp("M")

    plot_df = plot_df[
        (plot_df["cost_type"] == cost_type) &
        (plot_df["incurred_month"] <= end_month)
    ].copy()

    # IMPORTANT:
    # group by payment_installment, not offset_days
    summary = (
        plot_df
        .groupby(["cost_type", "payment_installment", "incurred_month"], as_index=False)["incurred_cost"]
        .sum()
    )

    pivot = (
        summary
        .pivot_table(
            index="incurred_month",
            columns="payment_installment",
            values="incurred_cost",
            aggfunc="sum",
            fill_value=0
        )
        .sort_index()
    )

    # Ensure all installment columns defined in PAYMENT_SCHEDULE are shown,
    # even if some are absent in the filtered data
    all_installments = sorted(
        PAYMENT_SCHEDULE.loc[
            PAYMENT_SCHEDULE["cost_type"] == cost_type,
            "payment_installment"
        ].unique()
    )

    pivot = pivot.reindex(columns=all_installments, fill_value=0)

    # Ensure all months up to Dec 2026 are shown
    if not pivot.empty:
        full_months = pd.date_range(
            start=pivot.index.min(),
            end=end_month,
            freq="M"
        )
        pivot = pivot.reindex(full_months, fill_value=0)
    else:
        # If no data at all, still create a monthly index ending Dec 2026
        full_months = pd.date_range(
            end=end_month,
            periods=1,
            freq="M"
        )
        pivot = pd.DataFrame(index=full_months, columns=all_installments).fillna(0)

    # Rename columns for legend
    pivot.columns = [f"Instalment {int(col)}" for col in pivot.columns]

    ax = pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 6)
    )

    ax.set_title(f"{cost_type.capitalize()} Incurred Cost by Month and Instalment")
    ax.set_xlabel("Incurred Month")
    ax.set_ylabel("Incurred Cost")
    ax.legend(title="Payment Instalment", bbox_to_anchor=(1.02, 1), loc="upper left")

    # X-axis labels as month-year
    ax.set_xticklabels([d.strftime("%b-%Y") for d in pivot.index], rotation=45, ha="right")

    plt.tight_layout()
    plt.show()


In [123]:
# plot_incurred_cost_by_month_and_installment(detail, cost_type="bom")

In [122]:
# plot_incurred_cost_by_month_and_installment(detail, cost_type="connection")

In [121]:
# plot_incurred_cost_by_month_and_installment(detail, cost_type="installation")

In [120]:
def draw_stacked_bar_chart(monthly_by_type: pd.DataFrame):
    monthly_total_by_cost_type = (
        monthly_by_type.groupby(["incurred_month", "cost_type"], as_index=False)["incurred_cost"]
        .sum()
        .pivot_table(
            index="incurred_month",
            columns="cost_type",
            values="incurred_cost",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
        .rename_axis(columns=None)
    )

    for col in ["bom", "connection", "installation"]:
        if col not in monthly_total_by_cost_type.columns:
            monthly_total_by_cost_type[col] = 0.0

    monthly_total_by_cost_type["total_capex"] = monthly_total_by_cost_type[
        ["bom", "connection", "installation"]
    ].sum(axis=1)
    monthly_total_by_cost_type = monthly_total_by_cost_type[
        ["incurred_month", "bom", "connection", "installation", "total_capex"]
    ]

    monthly_total_by_cost_type = monthly_total_by_cost_type.sort_values("incurred_month")
    plt.figure(figsize=(12, 6))
    plt.bar(
        monthly_total_by_cost_type["incurred_month"].dt.strftime("%Y-%m"),
        monthly_total_by_cost_type["bom"],
        label="BOM",
    )
    plt.bar(
        monthly_total_by_cost_type["incurred_month"].dt.strftime("%Y-%m"),
        monthly_total_by_cost_type["connection"],
        bottom=monthly_total_by_cost_type["bom"],
        label="Connection",
    )
    plt.bar(
        monthly_total_by_cost_type["incurred_month"].dt.strftime("%Y-%m"),
        monthly_total_by_cost_type["installation"],
        bottom=monthly_total_by_cost_type["bom"] + monthly_total_by_cost_type["connection"],
        label="Installation",
    )
    plt.xlabel("Incurred Month")
    plt.ylabel("Total Incurred CAPEX")
    plt.title("Monthly Incurred CAPEX by Cost Type")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# draw_stacked_bar_chart(monthly_by_type)

In [133]:
temp=monthly_by_type.groupby(["incurred_month"], as_index=False)["incurred_cost"].sum()
temp=temp[(temp.incurred_month < "2027-01-01")&(temp.incurred_month >= "2026-01-01")]
temp.incurred_cost.sum()

np.float64(25034531.233333338)

In [131]:
temp

,incurred_month,incurred_cost
0,2025-08-31,6.840190e+04
1,2025-09-30,3.600100e+03
2,2025-10-31,3.870108e+05
3,2025-11-30,3.286067e+05
4,2025-12-31,2.018522e+05
5,2026-01-31,7.080051e+05
6,2026-02-28,1.288381e+06
7,2026-03-31,1.469028e+06
8,2026-04-30,1.479914e+06
9,2026-05-31,1.219928e+06
